# Bike Share Toronto Ridership dataset 2024 (Toronto Open Data)

In [2]:
import pandas as pd 
import json
from pathlib import Path
RAW_DIR = Path("data/raw")

## Ridership Data

In [3]:
data_2024 = pd.read_csv(RAW_DIR / "bikeshare-ridership-2024.csv")

In [62]:
data_2024.head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC


In [8]:
data_2024.shape

(6953094, 11)

In [10]:
data_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 6953094 entries, 0 to 6953093
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    int64  
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(1), int64(4), str(6)
memory usage: 1.2 GB


## Initial data checks
Couple of things to check:
- convert the columns to appropriate data types
- split the data into monthly datasets
- check if trip_id has any duplicates
- check for all possible values for user_type and bike_model columns
- check for null values

In [16]:
data_2024["Trip_Id"].duplicated().sum() #check if there are any duplicates in trip_id col

np.int64(0)

In [13]:
data_2024["User_Type"].unique()

<ArrowStringArray>
['Casual', 'Member']
Length: 2, dtype: str

### BikeID to Model

In [5]:
models_per_bike = data_2024.groupby("Bike_Id")["Bike_Model"].nunique()

In [6]:
models_per_bike.shape

(8085,)

In [7]:
(models_per_bike > 1).sum()

np.int64(0)

mapping's clean

In [65]:
data_2024["Bike_Id"].nunique()

8085

In [14]:
data_2024["Bike_Model"].unique()

<ArrowStringArray>
['EFIT G5', 'ICONIC', 'EFIT']
Length: 3, dtype: str

### Station Id to Station Name

In [20]:
start_stations = (data_2024[["Start_Station_Id", "Start_Station_Name"]].drop_duplicates().rename(columns={"Start_Station_Id": "station_id", "Start_Station_Name": "station_name"}))

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [21]:
end_stations = (data_2024[["End_Station_Id", "End_Station_Name"]].dropna().drop_duplicates().rename(columns={"End_Station_Id": "station_id", "End_Station_Name": "station_name"}))

In [22]:
end_stations["station_id"] = end_stations["station_id"].astype("int64")

In [23]:
stations_all = pd.concat([start_stations, end_stations]).drop_duplicates()

In [26]:
#names per id
names_per_id = stations_all.groupby("station_id")["station_name"].nunique()
names_per_id.shape

(881,)

In [27]:
(names_per_id > 1).sum() #ids w more than one name

np.int64(0)

each id has only one name

### Missing Values

#### Null counts by column

In [17]:
data_2024.isna().sum()

Trip_Id                  0
Trip_Duration            0
Start_Station_Id         0
Start_Time               0
Start_Station_Name       0
End_Station_Id        8424
End_Time              5822
End_Station_Name      8424
Bike_Id                  0
User_Type                0
Bike_Model               0
dtype: int64

In [63]:
data_2024[data_2024.isna().any(axis=1)].head(3)


,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
20,26682765,4327,7881,2024-01-01 00:14:53,Queens Quay E / Merchants' Wharf,NaN,2024-01-01 01:27:00,NaN,3379,Casual,ICONIC
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC


In [64]:
data_2024[data_2024["End_Station_Id"].isna()].head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
20,26682765,4327,7881,2024-01-01 00:14:53,Queens Quay E / Merchants' Wharf,NaN,2024-01-01 01:27:00,NaN,3379,Casual,ICONIC
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC


In [65]:
data_2024[data_2024["End_Time"].isna()].head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
65291,26757586,0,7527,2024-01-11 16:56:33,Joseph J Piccininni Community Centre,NaN,NaN,NaN,5688,Casual,ICONIC
122444,26822518,0,7204,2024-01-21 18:17:57,College St / Crawford St,NaN,NaN,NaN,752,Member,ICONIC
157386,26863071,0,7029,2024-01-26 19:00:34,Bay St / Bloor St W (East Side),NaN,NaN,NaN,1297,Casual,ICONIC


We've got **8424** records where the "End_Station_Id" and "End_Station_Name" are missing, and **5822** records where the "End_Time" is missing (this could potentially be the subset of those 8424 records)

#### End_Station_Id and End_Station_Name

In [51]:
#check if missing End_Station_Id always corresponds to a missing End_Station_Name (this prob should)
#check if there are rows where the ID is missing but the Station name is not:
data_2024[(data_2024["End_Station_Id"].isna()) & (data_2024["End_Station_Name"].notna())].shape

(0, 11)

In [30]:
#check for rows where the Id exists but the name is missing:
data_2024[data_2024["End_Station_Id"].notna() & data_2024["End_Station_Name"].isna()].shape

(0, 11)

In [34]:
#or just do:
data_2024[data_2024["End_Station_Id"].isna() & data_2024["End_Station_Name"].isna()].shape


(8424, 11)

In [31]:
data_2024[["End_Station_Id", "End_Station_Name"]].isna().value_counts()

End_Station_Id  End_Station_Name
False           False               6944670
True            True                   8424
Name: count, dtype: int64

#### End_Time vs End station

In [36]:
#now check if records where the end_time is nan is a subset of those 8424 records
data_2024[data_2024["End_Time"].isna() & data_2024["End_Station_Id"].isna()].shape


(3313, 11)

Only **3313** of records where End_Time is null are a subset of records where **both** End_Station_Id and End_Station_Name are null

In [67]:
end_time_nan = data_2024[data_2024["End_Time"].isna() & data_2024["End_Station_Id"].notna()]
end_time_nan.shape

(2509, 11)

In [68]:
end_time_nan.head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
2678398,29756233,0,7190,2024-07-02 16:08:02,St. George St / Hoskin Ave,7176.0,NaN,Bathurst St / Fort York Blvd,6307,Casual,ICONIC
2678821,29756682,0,7409,2024-07-02 16:20:00,Isabella St / Church St,7028.0,NaN,Gould St / Mutual St,37,Member,ICONIC
2679069,29756947,0,7374,2024-07-02 16:27:38,Frederick St / The Esplanade,7028.0,NaN,Gould St / Mutual St,1782,Member,ICONIC


So it's not always that when End_Station_Id is null, the end_time is also null. Earlier assumption was wrong.

In [41]:
data_2024[["End_Station_Id", "End_Station_Name", "End_Time"]].isna().value_counts()


End_Station_Id  End_Station_Name  End_Time
False           False             False       6942161
True            True              False          5111
                                  True           3313
False           False             True           2509
Name: count, dtype: int64

### Trip_Duration Checks

In [55]:
data_2024[(data_2024["Trip_Duration"] == 0)].shape

(10873, 11)

In [69]:
data_2024[(data_2024["Trip_Duration"] == 0)].head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
980,26683830,0,7091,2024-01-01 11:12:50,Donlands Subway Station,7309.0,2024-01-01 11:12:50,Queen St E / Rhodes Ave,7398,Member,EFIT G5
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC


In [54]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())].shape

(5829, 11)

In [70]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())].head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
980,26683830,0,7091,2024-01-01 11:12:50,Donlands Subway Station,7309.0,2024-01-01 11:12:50,Queen St E / Rhodes Ave,7398,Member,EFIT G5
4182,26687522,0,7030,2024-01-02 08:52:47,Bay St / Wellesley St W,7713.0,2024-01-02 08:52:47,Temperance St / Bay St,2036,Member,ICONIC
9170,26693071,0,7059,2024-01-02 20:33:18,Front St W / Blue Jays Way,7010.0,2024-01-02 20:33:18,King St W / Spadina Ave,7316,Member,ICONIC


In [66]:
zero_with_end = data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())]
(zero_with_end["Start_Station_Id"] == zero_with_end["End_Station_Id"]).value_counts()

False    5276
True      553
Name: count, dtype: int64

553 -> 0 Time_Duration, and has the same start and end station  
5276 -> 0 Time_Duration, and different start and end station

In [58]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].isna())].shape

(5044, 11)

In [71]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].isna())].head(3)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC
4689,26688136,0,7359,2024-01-02 10:29:25,Rosedale Park (20 Scholfield Ave),NaN,2024-01-02 10:29:25,NaN,1790,Member,ICONIC


In [59]:
data_2024[(data_2024["Trip_Duration"] == 0) & ((data_2024["Start_Station_Id"].isna()) & (data_2024["End_Station_Id"].isna()))].shape

(0, 11)

10,873 trips have a Trip_Duration of 0 and Start_Time and End_Time equals for all of them. 3 groups:
- 5044: have no end station. These look like rentals that never became real trips
- 553: start and end at the same station. looks like bike being undocked and returned right away
- 5276: start and end at different stations, but a trip between two stationca cant take zero seconds => timestamps wrong? 

all three groups will be excluded from 'trips'

### Summary

What all needs cleaning:

- End Station and End time both missing: 3313 no desitination and no end, the trip never closed properly
- End station missing, end time present: 5111 has end time but no destination
- End time missing and station present: 2509 has a destination but no end time
- Zero duration, same start and end station: 553 undocked and returned right away
- Zero duration, different stations: 5276 impossible timing
- Zero duration, no end station: 5044 already inside the missing end station groups above

The count overlaps. Need to remove all the above rows in the cleaning step.

Other possible checks:
- duration vs timestamps: check both times and compare End_Time - Start_Time to Trip_Duration
- very short trips: trips, say under 60s between different stations
- very long trips: multiple day trips? bikes not docked properly

## Station Information Data

This is a snapshot from Oct 1, 2026. Not 2024

In [8]:
with open (RAW_DIR/"station_information.json") as file:
    stations = json.load(file)

In [9]:
snapshot_time = pd.to_datetime(stations["last_updated"], unit = "s")

In [19]:
snapshot_time #UTC time

Timestamp('2026-10-01 22:15:52')

In [11]:
stations_raw = pd.json_normalize(stations["data"]["stations"])

In [15]:
stations_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1073 entries, 0 to 1072
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   station_id              1073 non-null   str    
 1   external_id             1073 non-null   str    
 2   name                    1073 non-null   str    
 3   physical_configuration  1070 non-null   str    
 4   lat                     1073 non-null   float64
 5   lon                     1073 non-null   float64
 6   altitude                692 non-null    float64
 7   address                 1072 non-null   str    
 8   capacity                1073 non-null   int64  
 9   is_charging_station     1073 non-null   bool   
 10  rental_methods          1070 non-null   object 
 11  groups                  1073 non-null   object 
 12  obcn                    1073 non-null   str    
 13  short_name              1073 non-null   str    
 14  nearby_distance         1073 non-null   float64
 15

In [16]:
stations_raw["station_id"].nunique()

1073

In [17]:
stations_raw.isna().sum()

station_id                   0
external_id                  0
name                         0
physical_configuration       3
lat                          0
lon                          0
altitude                   381
address                      1
capacity                     0
is_charging_station          0
rental_methods               3
groups                       0
obcn                         0
short_name                   0
nearby_distance              0
_ride_code_support           0
post_code                  289
is_valet_station          1033
_valet_station_details    1056
cross_street               664
dtype: int64

In [12]:
stations_raw.shape

(1073, 20)

In [72]:
stations_raw.tail(3)

,station_id,external_id,name,physical_configuration,lat,lon,altitude,address,capacity,is_charging_station,rental_methods,groups,obcn,short_name,nearby_distance,_ride_code_support,post_code,is_valet_station,_valet_station_details,cross_street
1070,8269,3e1cc17a-55cd-43a5-af0a-ce38ea9c4753,Greystone Walk Dr / Midland Ave,VAULT,43.724207,-79.252430,0.0,Greystone Walk Dr / Midland Ave,11,False,"[key, transitcard, phone]","[East, Scarborough - Rouge Hill]",647-448-6545,647-448-6545,1000.0,True,M1K 5J4,NaN,NaN,Greystone Walk Dr / Midland Ave
1071,8271,d531d92c-dafc-44fb-878e-a3b68d512a76,R.C. Harris Treatment Plant,VAULT,43.674894,-79.278307,0.0,R.C. Harris Treatment Plant,19,False,"[key, transitcard, phone]","[East, Leslieville - Woodbine]",,,1000.0,True,M1N 1A5,NaN,NaN,Queen St E / Blantyre Ave
1072,8273,1a174532-c7ff-4e59-bbf6-f873de33bed8,Exhibition Place – Coca Cola Coliseum,REGULAR,43.634427,-79.415647,0.0,Exhibition Place – Coca Cola Coliseum,31,False,"[key, transitcard, creditcard, phone]","[West, Watefront]",,,1000.0,True,M6K 3C3,NaN,NaN,Princes' Blvd / Nunavut Rd


To clean
- station_id is a string. Convert to int
- Remove columns we dont need
- Names have double spaces, strip()
- Since the snapshot is from today => stations that were removed or renamed since wont be in it, and new ones will be. 

In [47]:
stations_raw["station_id"] = stations_raw["station_id"].astype("int64")

**stations_raw**= all stations from 2026  
**stations_all** = all stations from 2024

In [52]:
stations_raw[stations_raw["station_id"] == 7647]

,station_id,external_id,name,physical_configuration,lat,lon,altitude,address,capacity,is_charging_station,rental_methods,groups,obcn,short_name,nearby_distance,_ride_code_support,post_code,is_valet_station,_valet_station_details,cross_street


In [53]:
stations_all[stations_all["station_id"] == 7647]

,station_id,station_name
332,7647,Elm St/ University Ave (East Side)


#### Check for stations that are in 2026 but not in 2024

In [60]:
missing_2024 = stations_raw[~stations_raw["station_id"].isin(stations_all["station_id"])]
missing_2024.head(3)

,station_id,external_id,name,physical_configuration,lat,lon,altitude,address,capacity,is_charging_station,rental_methods,groups,obcn,short_name,nearby_distance,_ride_code_support,post_code,is_valet_station,_valet_station_details,cross_street
4,7004,f8ab24fe-e9c7-11ef-984e-124c66246c7d,University Ave / Elm St,VAULT,43.656499,-79.389362,0.0,University Ave / Elm St,35,False,"[key, transitcard, phone]",[P7004-7047],647-643-9673,647-643-9673,500.00,True,NaN,NaN,NaN,NaN
313,7382,f8c2ccd4-e9c7-11ef-984e-124c66246c7d,Simcoe St / Adelaide St W,VAULT,43.648130,-79.386320,NaN,Simcoe St / Adelaide St W,27,False,"[key, transitcard, phone]","[South, Financial District]",437-688-7395,437-688-7395,1000.00,True,NaN,NaN,NaN,NaN
460,7578,f8c5a2fc-e9c7-11ef-984e-124c66246c7d,Oak St / Parliament St,REGULAR,43.660810,-79.366238,0.0,Oak St / Parliament St,15,False,"[key, transitcard, creditcard, phone]",[],437-240-6919,437-240-6919,1609.34,True,M5A 2Z9,NaN,NaN,NaN


In [59]:
missing_2024.shape

(242, 20)

#### Check for stations that are in 2024 but not in 2026

In [61]:
missing_2026 = stations_all[~stations_all["station_id"].isin(stations_raw["station_id"])]
missing_2026.head(3)

,station_id,station_name
1,7245,Tecumseth St / Queen St W - SMART
20,7881,Queens Quay E / Merchants' Wharf
21,7724,Wellesley St E / Parliament St


In [57]:
missing_2026.shape

(50, 2)